# 01c — Decoder-Init Ablation Artifacts (derived from v5)

Builds 4 init artifacts that share v5's SALT **encoder verbatim** and differ ONLY in decoder weights.
Decoder **bias** (Vietnamese unigram log-frequency) is identical across all arms.

| arm | decoder weights | tests |
|---|---|---|
| `decscale05` | `E_salt @ M × 0.5` | is 0.1 starving gradients? |
| `decscale10` | `E_salt @ M × 1.0` | full-strength global map |
| `dectied` | copy of SALT embeddings (tied-at-init, trained untied) | conversation's "tied" arm, save/load-safe |
| `decpertoken` | per-token SALT maps → NeoBERT decoder rows, × 0.1 | historically-worse variant, retested post-NaN-fix |

Reference arm = **v5 itself** (global map × 0.1).

**Prereqs:** nb12 certified v5 finite on CUDA; v5 dir contains anchor CSVs, pruned tokenizer, `cc.vi.300.bin`.
**Compare arms by:** matched short CPT curves in nb02 (same seed/data/steps, fresh `RUN_NAME` per arm) + `salt3_init_signal_probe` — **NOT by step-0 loss** (bias-dominated by design).

In [ ]:
%%capture
!pip install -U transformers accelerate safetensors sentencepiece tokenizers pandas tqdm huggingface_hub fasttext-wheel

In [ ]:
import sys, json, shutil, glob
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F

try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print('Drive mount skipped:', e)

PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
sys.path.insert(0, str(PROJECT_ROOT / 'code'))
import importlib
import salt3_common as sc; importlib.reload(sc)
import salt3_decoder_variants as sdv; importlib.reload(sdv)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

V5_NAME  = 'videberta_salt_init_v5_globalmap_freqbias'
V5_DIR   = PROJECT_ROOT / 'init' / V5_NAME
V5_MODEL = V5_DIR / 'model'
VARIANT_PREFIX = 'videberta_salt_init_v5'

ARMS = {
    'decscale05':  dict(decoder_init='global_emb_to_decoder_map',      decoder_weight_scale=0.5),
    'decscale10':  dict(decoder_init='global_emb_to_decoder_map',      decoder_weight_scale=1.0),
    'dectied':     dict(decoder_init='tied_to_embeddings',             decoder_weight_scale=None),
    'decpertoken': dict(decoder_init='projected_from_neobert_decoder', decoder_weight_scale=0.1),
}
def arm_dir(arm):
    return PROJECT_ROOT / 'init' / f'{VARIANT_PREFIX}_{arm}'

v5_cfg = json.loads((V5_DIR / 'salt_config.json').read_text(encoding='utf-8'))
V5_SCALE = float(v5_cfg.get('decoder_weight_scale', 0.1) or 0.1)
print('v5 decoder scale :', V5_SCALE)
print('torch', torch.__version__, '| device', DEVICE)
for arm in ARMS:
    print('will build:', arm_dir(arm).name)

## A. Scale + tied arms (pure tensor surgery, no GPU needed)

v5's saved decoder is `E_salt @ M × 0.1`; dividing out 0.1 recovers `E_salt @ M` exactly, then the
target scale is applied. The tied arm copies the embedding matrix — tied **at init only**: true
weight-sharing is deliberately avoided because shared tensors get dropped from safetensors
checkpoints (the old x2_tied artifact lost its decoder keys exactly this way and silently loaded a
random head). Tied-at-init also keeps every arm's training dynamics identical, so the comparison
isolates the init strategy.

In [ ]:
from safetensors.torch import load_file

state  = load_file(str(V5_MODEL / 'model.safetensors'))
dec_v5 = state['decoder.weight'].float()        # = E_salt @ M * V5_SCALE
emb_v5 = state['model.encoder.weight'].float()
print('v5 decoder:', tuple(dec_v5.shape), '| emb:', tuple(emb_v5.shape))

for arm in ('decscale05', 'decscale10'):
    scale = ARMS[arm]['decoder_weight_scale']
    w = dec_v5 * (scale / V5_SCALE)
    sdv.write_variant(V5_DIR, arm_dir(arm), w, {'init_name': arm_dir(arm).name, **ARMS[arm]})

sdv.write_variant(V5_DIR, arm_dir('dectied'), emb_v5.clone(),
                  {'init_name': arm_dir('dectied').name, **ARMS['dectied']})

## B. Per-token SALT decoder arm

Refits SALT Eq.3 local maps with **NeoBERT decoder rows** as regression targets (instead of
embedding rows): `X = pinv(A_vi_emb) @ D_en`, `d_t = e_t_vi @ X`. Anchors/specials copy source
decoder rows directly. Same ×0.1 scale and same freq bias as v5, so the only difference vs v5 is
per-token vs global construction. This arm scored ~15 step-0 loss historically — it is retested
here because that number predates the NaN fix and the freq-bias decoder.

In [ ]:
import fasttext
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

# Donor embeddings: full ViDeBERTa table (GDES: the trained tensor is `weight`, not `_weight`)
videberta = AutoModel.from_pretrained('Fsoft-AIC/videberta-base')
vi_emb_full = sc.extract_embedding_weight(videberta).float().cpu()
del videberta
print('ViDeBERTa emb   :', tuple(vi_emb_full.shape))

# NeoBERT decoder rows straight from hub safetensors — no model instantiation,
# so no xformers check_imports requirement.
neo_st = load_file(hf_hub_download('chandar-lab/NeoBERT', 'model.safetensors'))
src_decoder = neo_st['decoder.weight'].float()
print('NeoBERT decoder :', tuple(src_decoder.shape))

source_tokenizer = sc.load_tokenizer_no_remote_code('chandar-lab/NeoBERT', V5_DIR / 'neobert_tokenizer_local')
source_vocab = source_tokenizer.get_vocab()
target_tokenizer = AutoTokenizer.from_pretrained(V5_MODEL, trust_remote_code=True)

full_tok_json = hf_hub_download('Fsoft-AIC/videberta-base', 'tokenizer.json')
target_vocab, new_to_old = sdv.rebuild_vocab_maps(V5_DIR / 'pruned_videberta_tokenizer', full_tok_json)
anchor_map = sdv.read_anchor_map(V5_DIR)
specials = sdv.special_token_pairs(target_tokenizer, source_tokenizer, target_vocab, source_vocab)
print(f'vocab={len(target_vocab)} anchors={len(anchor_map)} specials={len(specials)}')
assert len(target_vocab) == emb_v5.shape[0], 'pruned vocab != v5 embedding rows'
assert len(anchor_map) > 3000, 'anchor CSVs incomplete — check v5 dir'

FT_BIN = V5_DIR / 'cc.vi.300.bin'
if not FT_BIN.exists():
    import gzip, urllib.request
    print('fastText bin missing — downloading...')
    urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.vi.300.bin.gz', str(FT_BIN) + '.gz')
    with gzip.open(str(FT_BIN) + '.gz', 'rb') as fi, open(FT_BIN, 'wb') as fo:
        shutil.copyfileobj(fi, fo)
ft = fasttext.load_model(str(FT_BIN))
ft_vec = lambda tok: ft.get_word_vector(tok.replace('\u2581', ''))

dec_pt, stats = sdv.build_pertoken_decoder(
    vi_emb_full, src_decoder, target_vocab, new_to_old, source_vocab,
    anchor_map, specials, ft_vec, device=DEVICE, min_anchors=8)
print(stats)

sdv.write_variant(V5_DIR, arm_dir('decpertoken'),
                  dec_pt * ARMS['decpertoken']['decoder_weight_scale'],
                  {'init_name': arm_dir('decpertoken').name, **ARMS['decpertoken']})

## C. Certify — CUDA forward finite per arm

Same protocol as nb12 cell C (purge dynamic-module cache, fresh import, masked-MLM loss on a fixed
seed-0 mask so losses are comparable across arms). Reminder: these step-0 losses are
**bias-dominated diagnostics**, not the ranking metric — rank arms by matched nb02 CPT curves.

In [ ]:
import importlib as _il
from transformers import AutoModelForMaskedLM, AutoTokenizer

for c in glob.glob('/root/.cache/huggingface/modules/transformers_modules/*'):
    if Path(c).is_dir():
        shutil.rmtree(c, ignore_errors=True)
for k in [k for k in sys.modules if k.startswith('transformers_modules')]:
    del sys.modules[k]
_il.invalidate_caches()

@torch.no_grad()
def mlm_check(model_dir):
    m = AutoModelForMaskedLM.from_pretrained(model_dir, trust_remote_code=True).to(DEVICE).eval()
    t = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    s = ['Việt Nam là một quốc gia ở Đông Nam Á.',
         'Hôm nay thời tiết rất đẹp và trời trong xanh.',
         'Kinh tế Việt Nam tăng trưởng trong năm qua.',
         'Trẻ em cần được tiêm phòng đầy đủ để tránh bệnh.']
    enc = t(s, padding=True, truncation=True, max_length=32, return_tensors='pt').to(DEVICE)
    ids = enc['input_ids']
    torch.manual_seed(0)
    pm = torch.full(ids.shape, 0.2)
    for sid in set(t.all_special_ids):
        pm[ids.cpu() == sid] = 0
    msk = torch.bernoulli(pm).bool().to(DEVICE)
    lab = torch.full_like(ids, -100); lab[msk] = ids[msk]
    mids = ids.clone(); mids[msk] = t.mask_token_id
    wfin = all(bool(torch.isfinite(p).all()) for p in m.parameters())
    lg = m(input_ids=mids, attention_mask=enc['attention_mask']).logits
    fin = bool(torch.isfinite(lg).all())
    loss = F.cross_entropy(lg.reshape(-1, lg.size(-1)).float(), lab.reshape(-1), ignore_index=-100).item()
    del m; torch.cuda.empty_cache()
    return fin, wfin, loss

rows = [('v5_globalmap_scale01 (ref)', V5_MODEL)] + [(arm_dir(a).name, arm_dir(a) / 'model') for a in ARMS]
print(f"{'artifact':52s} finite wfinite loss")
allok = True
for name, d in rows:
    fin, wfin, loss = mlm_check(d)
    allok &= fin
    print(f'{name:52s} {str(fin):6s} {str(wfin):6s}  {loss:.3f}')
print('=' * 74)
print('ALL ARMS FINITE — for each arm run nb02 with MODE=new, fresh RUN_NAME, matched steps/seed.'
      if allok else 'SOME ARM NON-FINITE — inspect before training.')